# ACSIncome utility check: original vs numeric coarsening

Local diagnostic using only `environment/task_inputs/train.npz` and `val.npz`. The validation archive is Florida 2018. No hidden data or labels are loaded. This does not modify packaged archives.


In [2]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path.cwd()
if not (root / "environment/task_inputs/train.npz").is_file():
    root = root.parents[1] if root.name == "reencoding" else root
train_path = root / "environment/task_inputs/train.npz"
val_path = root / "environment/task_inputs/val.npz"

with np.load(train_path, allow_pickle=False) as d:
    X_train = d["X"].copy()
    y_train = d["y"].astype(np.uint8, copy=True)
    train_features = d["feature_names"].tolist()
with np.load(val_path, allow_pickle=False) as d:
    X_val = d["X"].copy()
    y_val = d["y"].astype(np.uint8, copy=True)
    val_features = d["feature_names"].tolist()

assert X_train.shape == (434610, 10)
assert X_val.shape == (98925, 10)
assert train_features == val_features
assert np.isin(y_train, [0, 1]).all() and np.isin(y_val, [0, 1]).all()
print("Train X/y:", X_train.shape, y_train.shape)
print("Validation X/y:", X_val.shape, y_val.shape)
print("Feature order:", train_features)
print("Train/validation positive rates:", round(float(y_train.mean()), 4), round(float(y_val.mean()), 4))


Train X/y: (434610, 10) (434610,)
Validation X/y: (98925, 10) (98925,)
Feature order: ['AGEP', 'COW', 'SCHL', 'MAR', 'OCCP', 'POBP', 'RELP', 'WKHP', 'SEX', 'RAC1P']
Train/validation positive rates: 0.3972 0.3313


## Coarsening policy

Use the matching-audit policy without fitting boundaries on validation data: `AGEP` uses five-year bands starting at 17, represented by each band's midpoint; `WKHP` is rounded to the nearest five hours and clipped to 1–98. Code 99 is preserved as its top-coded value. All other features and both label arrays remain unchanged.


In [4]:
def coarsen_values(values, feature):
    values = np.asarray(values, dtype=float)
    result = values.copy()
    if feature == "AGEP":
        ordinary = values < 99
        lower = 17 + 5 * np.floor((values[ordinary] - 17) / 5)
        upper = np.minimum(lower + 4, 98)
        result[ordinary] = (lower + upper) / 2
    elif feature == "WKHP":
        ordinary = values < 99
        result[ordinary] = np.clip(np.floor(values[ordinary] / 5 + 0.5) * 5, 1, 98)
    else:
        raise ValueError(f"No tested coarsening rule for {feature}")
    return result

def coarsen_matrix(X, feature_names):
    result = X.copy()
    for feature in ("AGEP", "WKHP"):
        col = feature_names.index(feature)
        result[:, col] = coarsen_values(result[:, col], feature)
    return result

y_train_before = y_train.copy()
y_val_before = y_val.copy()
X_train_before = X_train.copy()
X_val_before = X_val.copy()
X_train_coarse = coarsen_matrix(X_train, train_features)
X_val_coarse = coarsen_matrix(X_val, val_features)

changed_columns = [i for i in range(X_train.shape[1]) if not np.array_equal(X_train[:, i], X_train_coarse[:, i])]
assert changed_columns == [train_features.index("AGEP"), train_features.index("WKHP")]
assert np.array_equal(y_train, y_train_before) and np.array_equal(y_val, y_val_before)
assert np.array_equal(X_train, X_train_before) and np.array_equal(X_val, X_val_before)
for name, X in [("train", X_train_coarse), ("validation", X_val_coarse)]:
    for feature in ("AGEP", "WKHP"):
        col = train_features.index(feature)
        assert np.array_equal(X[:, col] == 99, (X_train if name == "train" else X_val)[:, col] == 99)
print("Changed features:", [train_features[i] for i in changed_columns])
print("Label arrays unchanged; row counts and feature order preserved.")
print("Policy probes:")
print("AGEP:", coarsen_values(np.array([17, 21, 22, 26, 27, 98, 99]), "AGEP").tolist())
print("WKHP:", coarsen_values(np.array([1, 2, 3, 97, 98, 99]), "WKHP").tolist())


Changed features: ['AGEP', 'WKHP']
Label arrays unchanged; row counts and feature order preserved.
Policy probes:
AGEP: [19.0, 19.0, 24.0, 24.0, 29.0, 97.5, 99.0]
WKHP: [1.0, 1.0, 5.0, 95.0, 98.0, 99.0]


## Fixed model and metrics

Use the same logistic-regression pipeline, preprocessing, hyperparameters, random seed, and 0.5 probability threshold for both runs. One-hot encoding is used for the seven nominal categorical features; `SCHL` is kept numeric/ordinal with the two coarsened numeric features and standardized. Training time measures `.fit()` only.

ECE convention: **10-bin top-label ECE**. For each record, confidence is the probability assigned to the predicted class (`p` when `p >= 0.5`, otherwise `1-p`); bin accuracy is the fraction classified correctly. Bins are equal-width over `[0,1]`, with confidence 1 included in the last bin. This is not positive-class calibration ECE.


In [6]:
categorical_features = ["COW", "MAR", "OCCP", "POBP", "RELP", "SEX", "RAC1P"]
nominal_cols = [train_features.index(f) for f in categorical_features]
numeric_cols = [train_features.index(f) for f in ("AGEP", "SCHL", "WKHP")]

def make_model():
    prep = ColumnTransformer([
        ("nominal", OneHotEncoder(handle_unknown="ignore"), nominal_cols),
        ("numeric", StandardScaler(), numeric_cols),
    ])
    clf = LogisticRegression(C=1.0, max_iter=400, solver="lbfgs", random_state=42)
    return make_pipeline(prep, clf)

def top_label_ece(y_true, probabilities, n_bins=10):
    p = np.asarray(probabilities)
    pred = (p >= 0.5).astype(np.uint8)
    confidence = np.where(pred == 1, p, 1 - p)
    correct = (pred == y_true)
    bin_ids = np.minimum((confidence * n_bins).astype(int), n_bins - 1)
    ece = 0.0
    details = []
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.any():
            acc_b = float(correct[mask].mean())
            conf_b = float(confidence[mask].mean())
            ece += float(mask.mean()) * abs(acc_b - conf_b)
            details.append((b, int(mask.sum()), acc_b, conf_b))
    return ece, details

def evaluate_run(name, Xtr, Xva):
    model = make_model()
    start = time.perf_counter()
    model.fit(Xtr, y_train)
    fit_seconds = time.perf_counter() - start
    p_val = model.predict_proba(Xva)[:, 1]
    y_pred = (p_val >= 0.5).astype(np.uint8)
    accuracy = accuracy_score(y_val, y_pred)
    ece, bins = top_label_ece(y_val, p_val, n_bins=10)
    iterations = int(model.named_steps["logisticregression"].n_iter_[0])
    converged = iterations < 400
    print(f"{name}: fit={fit_seconds:.2f}s, iterations={iterations}, converged={converged}, Florida accuracy={accuracy:.6f}, top-label ECE={ece:.6f}")
    if not converged:
        raise RuntimeError(f"{name} model reached max_iter without convergence")
    return {"run":name, "accuracy":accuracy, "ece":ece, "fit_seconds":fit_seconds}, model

original_result, original_model = evaluate_run("Original", X_train, X_val)
coarse_result, coarse_model = evaluate_run("Coarsened", X_train_coarse, X_val_coarse)


Original: fit=3.54s, iterations=151, converged=True, Florida accuracy=0.787587, top-label ECE=0.007546
Coarsened: fit=3.41s, iterations=147, converged=True, Florida accuracy=0.787253, top-label ECE=0.007781


In [7]:
delta_accuracy = coarse_result["accuracy"] - original_result["accuracy"]
delta_ece = coarse_result["ece"] - original_result["ece"]
comparison = pd.DataFrame([
    {"Metric":"Florida accuracy", "Original":original_result["accuracy"], "Coarsened":coarse_result["accuracy"], "Difference (coarsened - original)":delta_accuracy},
    {"Metric":"10-bin top-label ECE", "Original":original_result["ece"], "Coarsened":coarse_result["ece"], "Difference (coarsened - original)":delta_ece},
    {"Metric":"Training time (seconds)", "Original":original_result["fit_seconds"], "Coarsened":coarse_result["fit_seconds"], "Difference (coarsened - original)":coarse_result["fit_seconds"]-original_result["fit_seconds"]},
])
print(comparison.to_string(index=False, formatters={
    "Original":lambda x:f"{x:.6f}", "Coarsened":lambda x:f"{x:.6f}",
    "Difference (coarsened - original)":lambda x:f"{x:+.6f}"
}))
print("\nAccuracy difference in percentage points:", round(delta_accuracy * 100, 4))
print("ECE difference in percentage points:", round(delta_ece * 100, 4))
print("Validation probability convention: p(y=1); prediction threshold = 0.5; ECE = top-label, 10 equal-width bins.")
print("Only visible train/Florida validation archives were used; no hidden data or labels were loaded.")
print("No packaged archive was written or changed.")


                 Metric Original Coarsened Difference (coarsened - original)
       Florida accuracy 0.787587  0.787253                         -0.000334
   10-bin top-label ECE 0.007546  0.007781                         +0.000235
Training time (seconds) 3.544195  3.411875                         -0.132320

Accuracy difference in percentage points: -0.0334
ECE difference in percentage points: 0.0235
Validation probability convention: p(y=1); prediction threshold = 0.5; ECE = top-label, 10 equal-width bins.
Only visible train/Florida validation archives were used; no hidden data or labels were loaded.
No packaged archive was written or changed.


## Interpretation

Use the measured accuracy and calibration changes to assess the predictive cost of coarsening. This is a local logistic-regression diagnostic, not the submission GPU oracle. It does not establish a privacy guarantee; the previous linkage audit found only a small matching-risk change under coarsening. Do not replace packaged archives on the basis of this experiment alone.
